# Phishing Website Detection Using Machine Learning
### CLA-3 Case Study | Google Colab Notebook

**Objective:** Build and compare machine-learning classifiers for distinguishing phishing websites from legitimate websites using the UCI Phishing Websites dataset.

**Important:** Run the notebook from top to bottom. Keep the output cells because the final report must contain results actually produced by this experiment. Do not copy example or assumed scores.

## 1. Install and import libraries

This notebook uses `ucimlrepo` to retrieve the dataset directly from the UCI Machine Learning Repository.

In [ ]:
!pip -q install ucimlrepo

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

RANDOM_STATE = 42
print("Libraries imported successfully.")

## 2. Load the UCI dataset

Dataset: **Phishing Websites**, UCI Machine Learning Repository, dataset ID 327.

The code below loads the features and target from UCI. It prints the actual dimensions and target labels returned, rather than assuming a particular class distribution.

In [ ]:
phishing_data = fetch_ucirepo(id=327)

X = phishing_data.data.features.copy()
y = phishing_data.data.targets.copy()

# Ensure the target is a one-dimensional Series.
if isinstance(y, pd.DataFrame):
    if y.shape[1] != 1:
        raise ValueError(f"Expected one target column; found {y.shape[1]}. Inspect the dataset before proceeding.")
    y = y.iloc[:, 0]
else:
    y = pd.Series(y)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeature names:")
print(list(X.columns))
print("\nTarget values and counts:")
print(y.value_counts(dropna=False))

## 3. Inspect and understand the data

Check data types, missing values, duplicate rows, summary statistics, and target balance before preprocessing.

In [ ]:
display(X.head())
print("\nData types:")
display(X.dtypes.value_counts().rename("number of columns").to_frame())

print("\nMissing values in features:", int(X.isna().sum().sum()))
print("Missing values in target:", int(y.isna().sum()))
print("Duplicate feature rows:", int(X.duplicated().sum()))

display(X.describe(include="all").T)

In [ ]:
plt.figure(figsize=(7, 4))
y.astype(str).value_counts().plot(kind="bar")
plt.title("Distribution of target classes")
plt.xlabel("Original target label")
plt.ylabel("Number of records")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Prepare the data

Rows with missing target labels are removed. Missing feature values, if any, are imputed using the median. Non-numeric feature columns are one-hot encoded. The original target labels are retained and encoded for model training.

**Label interpretation:** The notebook prints the original target labels. Confirm their meaning using the UCI dataset documentation before writing the report. Do not assume the meaning of a numeric label without checking the dataset description.

In [ ]:
# Remove records without a known target.
valid_target = y.notna()
X = X.loc[valid_target].copy()
y = y.loc[valid_target].copy()

# Convert feature columns to numeric where possible; retain categorical columns otherwise.
X = pd.get_dummies(X, dummy_na=True)

# Replace infinite values and impute any missing feature values.
X = X.replace([np.inf, -np.inf], np.nan)
X = X.apply(lambda col: col.fillna(col.median()) if col.isna().any() else col)

# Encode target labels while retaining a mapping to their original values.
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y.astype(str))
class_names = [str(v) for v in label_encoder.classes_]

print("Prepared feature matrix:", X.shape)
print("Number of target classes:", len(class_names))
print("Encoded class mapping:")
for i, name in enumerate(class_names):
    print(f"  {i} -> original label {name}")

if len(class_names) < 2:
    raise ValueError("The target contains fewer than two classes; classification cannot proceed.")

## 5. Split into training and testing sets

A stratified 80:20 split preserves the class proportions in both sets. The test set is held out for final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])
print("Number of input features:", X_train.shape[1])

## 6. Train four classification algorithms

We compare Logistic Regression, Decision Tree, Random Forest, and Support Vector Machine (SVM).

Scaling is included in pipelines for Logistic Regression and SVM. Decision Tree and Random Forest do not require feature scaling. The pipelines ensure the scaler is fitted only on training data, avoiding data leakage.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Support Vector Machine": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC())
    ])
}

trained_models = {}
predictions = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model
    predictions[name] = model.predict(X_test)

print("\nAll four models have been trained.")

## 7. Evaluate and compare the models

For multiclass or binary classification, weighted precision, recall, and F1-score are reported. Accuracy is also shown. Weighted averages account for class support.

Use the scores generated by your own run in the case-study report.

In [ ]:
results = []

for name, y_pred in predictions.items():
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (weighted)": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall (weighted)": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1-score (weighted)": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1-score (weighted)", ascending=False).reset_index(drop=True)
display(results_df.style.format({
    "Accuracy": "{:.4f}",
    "Precision (weighted)": "{:.4f}",
    "Recall (weighted)": "{:.4f}",
    "F1-score (weighted)": "{:.4f}"
}))

best_model_name = results_df.loc[0, "Model"]
print("Best model by weighted F1-score:", best_model_name)

In [ ]:
plt.figure(figsize=(10, 5))
plot_df = results_df.set_index("Model")[["Accuracy", "Precision (weighted)", "Recall (weighted)", "F1-score (weighted)"]]
plot_df.plot(kind="bar", figsize=(11, 5))
plt.title("Performance comparison of classification algorithms")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=20, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 8. Confusion matrices and detailed reports

A confusion matrix shows how many examples of each actual class were predicted as each class. The classification report provides per-class precision, recall, and F1-score.

In [ ]:
for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names
    )
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted original target label")
    plt.ylabel("Actual original target label")
    plt.tight_layout()
    plt.show()

    print(f"Classification report — {name}")
    print(classification_report(
        y_test, y_pred, target_names=class_names, zero_division=0
    ))

## 9. Inspect correct and incorrect predictions

The following cell displays examples from the held-out test set for the best model. The rows are examples from the dataset, not live website scans. The original target labels are shown; verify their meaning in the UCI documentation before describing them as phishing or legitimate.

In [ ]:
best_model = trained_models[best_model_name]
best_predictions = predictions[best_model_name]

examples = X_test.copy()
examples["Actual encoded class"] = y_test
examples["Predicted encoded class"] = best_predictions
examples["Actual original label"] = label_encoder.inverse_transform(y_test)
examples["Predicted original label"] = label_encoder.inverse_transform(best_predictions)
examples["Correct prediction"] = (y_test == best_predictions)

correct_examples = examples[examples["Correct prediction"]].head(3)
incorrect_examples = examples[~examples["Correct prediction"]].head(3)

print("Correctly classified examples:")
display(correct_examples)

print("Incorrectly classified examples:")
if len(incorrect_examples) > 0:
    display(incorrect_examples)
else:
    print("No incorrect examples were produced on this test set by the selected model.")

## 10. Feature importance (tree-based models)

For Decision Tree and Random Forest, feature importance estimates how useful each feature was to the fitted trees. This is not proof of causation.

In [ ]:
rf_model = trained_models["Random Forest"]
rf_importance = pd.Series(
    rf_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False).head(15)

plt.figure(figsize=(9, 6))
rf_importance.sort_values().plot(kind="barh")
plt.title("Top 15 feature importances — Random Forest")
plt.xlabel("Feature importance")
plt.tight_layout()
plt.show()

display(rf_importance.to_frame("Importance"))

## 11. Save the best model and experiment outputs

The model is saved as a local notebook-session file. Download it from the Colab Files panel if you want to keep it. The results table is also exported as CSV.

In [ ]:
import joblib

joblib.dump({
    "model": trained_models[best_model_name],
    "label_encoder": label_encoder,
    "feature_columns": list(X_train.columns),
    "model_name": best_model_name
}, "phishing_detection_best_model.joblib")

results_df.to_csv("phishing_model_comparison_results.csv", index=False)

print("Saved: phishing_detection_best_model.joblib")
print("Saved: phishing_model_comparison_results.csv")

## 12. Final experiment summary

Run this cell after all previous cells. Use the printed information to write the Results and Discussion section. Do not claim that this experiment scans arbitrary live URLs: the model was trained and tested on the dataset's supplied features.

In [ ]:
print("CASE STUDY EXPERIMENT SUMMARY")
print("=" * 45)
print("Dataset: UCI Phishing Websites (ID 327)")
print("Prepared records:", len(X))
print("Input features after encoding:", X.shape[1])
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Target labels (original):", class_names)
print("\nModel comparison:")
display(results_df)
print("\nBest model by weighted F1-score:", best_model_name)
print("\nInterpretation reminder:")
print("Confirm the original target-label meanings from UCI documentation.")
print("Report only the scores and examples produced by this notebook run.")

## References to include in the report

1. UCI Machine Learning Repository, **Phishing Websites** dataset, dataset ID 327. https://archive.ics.uci.edu/dataset/327/phishing+websites
2. Scikit-learn documentation, **Classification metrics**. https://scikit-learn.org/stable/modules/model_evaluation.html
3. Scikit-learn documentation, **Pipeline**. https://scikit-learn.org/stable/modules/compose.html#pipeline

**Base research paper:** Add the full bibliographic details of the research paper you have personally checked and used. Verify its authors, title, year, journal, and DOI before submitting. Do not cite a paper solely because it appeared in an earlier draft.